This notebook implements dimensionality reduction for mutational series. 

Options include:
```
    - UMAP (Fig. 1B; left) + sfGFP distributions (Fig. 1B; right)
    - PCA (not included in the paper)
    - t-SNE (PCA initiated; not included in the paper)
```

The imported dataset is a cleaned and reduced version from [Cambray et al., 2018](https://www.nature.com/articles/nbt.4238).

Specifically:
```
    mut_series (->int): mutational series indexes as per the original dataset
    Combi      (->int): combination of eight sequence properties previously found to correlate with protein production in E. coli
    rep        (->int): DNA sequence variant identifier containing between one and four mutations
    Sequence   (->str): 96nt long DNA sequence
    Protein  (->float): arithmetic mean of sfGFP fluorescence across 3 replicates for the case of normal translational initiation
    cdsCAI   (->float): codon adaptation index
    utrCdsStructureMFE    (->float): stability of 1st tile's secondary structure
    fivepCdsStructureMFE  (->float): stability of 2nd tile's secondary structure
    threepCdsStructureMFE (->float): stability of 3rd tile's secondary structure
    cdsBottleneckPosition (->int): codon ramp bottleneck position
    cdsBottleneckRelativeStrength (->float): codon ramp bottleneck strength
    cdsNucleotideContentAT (->float): A+T content (%AT)
    cdsHydropathyIndex (->float): mean hydrophobicity index

```





## 0. Mount Drive and set the main path

In [ ]:
#mount to drive
from google.colab import drive
try:
  drive.mount('/content/drive', force_remount=True)
except:
  drive._mount('/content/drive', force_remount=True)

# construct the path to use
import os
 
# get the current working directory
dir_path = os.path.dirname(os.path.realpath('__file__'))

# loop to crawl over your Drive and construct the correct path
for root, dirs, files in os.walk(dir_path):
    for file in files:
 
        # do not change the extension *.ipynb*
        if file.endswith('1_kmers.ipynb'):
          # check that we're using the correct parent directory
          if f'{root}/{file}'[-24:] == '/seq2yield/1_kmers.ipynb':
            # create path to the import directory
            drive_dir = f"{f'{root}/{file}'[:-13]}to_import/"

drive_dir

## 1. Import libraries, custom functions, and get data



In [ ]:
import os
import re
import time
import pickle
import random
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, drive_dir)

import utils, kMers

# load .csv into pandas dataframe
pd.options.display.max_rows = 10
raw_data = pd.read_csv(f'{drive_dir}Ecoli_data.csv', index_col= 0).dropna().reset_index(drop = True)

## 2. Dimensionality reduction for mutational series

Choose a subsample of the dataset based on the number of mutational series and the number of sequences needed, build a "corpus" in the variable "DATA" (->list of arrays)'

In [ ]:
# increase num to use more sequences
# change series to use different subsamples of the dataset
# change *k* to split sequences into k-mers of different lengths
num, series, k = 2400, 56, 4
series_dict = kMers.get_X_seqs(utils.x_split_seeds(raw_data, num, series), k=k)
corpus = kMers.seed_corpus(series_dict)

# get # of counts per k-mer
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

lst_of_arr = [cv.fit_transform(corpus[0]).toarray()]
for i in range(1, len(series_dict)):
  lst_of_arr.append(cv.transform(corpus[i]).toarray())

# central data structure to pass as input
# to dimensionality reduction methods
DATA = lst_of_arr[0]
label_vec = list(itertools.chain(*[list(series_dict[f'seed_{sr}'].seed.values) for sr in range(1,57)]))
for i in range(1, len(series_dict)):
  DATA = np.concatenate((DATA, lst_of_arr[i]), axis=0)

### 2.1 UMAP

**Note**: you may notice an error about potential dependency conflicts because *pip*'s dependency resolver does not take into account all packages installed. The code block **will run** as intended.

In [ ]:
try:
  import umap.plot
except:
  !pip -q install umap-learn[plot]
  import umap.plot

# Run UMAP
mapper = umap.UMAP(
    densmap=True,
    n_neighbors=15,
    min_dist=0,
    n_components=2,
    low_memory=True,
).fit(DATA)

*mapper* object contains the UMAP embeddings

1.   Create custom colormap for selected mutational series of Figs 1B & S1
2.   Plot 2D projection



In [ ]:
# list of series for Fig 1B
sr_to_plot = [4, 24, 39, 44, 52]
plot_labels = [elem if elem in sr_to_plot else 'none' for elem in label_vec]

# generate a palette and assign desired colours
ql = sns.cubehelix_palette(start=0.5, rot=-2)
cust_color = [[0,0,0], 
              ql[1], 
              ql[2], 
              ql[3], 
              ql[4], 
              ql[5]]

# plot
fig = plt.figure(figsize=(10, 10))
sns.scatterplot(mapper.embedding_[:,0], 
                mapper.embedding_[:,1], 
                hue     = plot_labels, 
                palette = ql, 
                alpha   = 0.2, 
                s = 6, 
                linewidths = 0.01, 
                legend=False).plot();

Finally, plot GFP expression distributions for the mutational series of Fig 1C

In [ ]:
empty_df, sr_list = pd.DataFrame(), ['seed_4', 'seed_24', 'seed_39', 'seed_44', 'seed_52']
for elm in sr_list:
  empty_df = pd.concat((empty_df, series_dict[elm]))

ql = sns.cubehelix_palette(start=0.5, rot=-2)
ql = ql[1:]
for idx, elm in enumerate(empty_df.mut_series.unique()):
  plt.subplots(figsize=(5,3))
  sns.histplot(data=empty_df[empty_df.mut_series == elm], 
               x="Protein", 
               color=ql[idx],
               kde=True, 
               bins = 28)
  plt.title(f'Mutational series {elm} | {sr_list[idx]}')
  plt.ylim((0, 350))

---
---


### 2.2 PCA (not included in the paper)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
embedded = pca.fit_transform(DATA)

fig = plt.figure(figsize=(15, 5))
sns.scatterplot(embedded[:,0], 
                embedded[:,1], 
                hue=label_vec, 
                alpha=1, 
                s=10).plot()  
plt.legend().set_visible(False)

### 2.3 t-SNE (PCA-initiated; not included in the paper)
*Note: this takes time to run; feel free to skip.*

In [ ]:
from sklearn.manifold import TSNE

embedded = TSNE(n_components=2, init='pca').fit_transform(DATA)

fig = plt.figure(figsize=(16, 10))
sns.scatterplot(embedded[:,0], 
                embedded[:,1], 
                hue=label_vec, 
                alpha=1, 
                s=0.5).plot()  
plt.legend().set_visible(False)